> **Synthetic test data that actually catches bugs. No production samples needed.**

# Data Generation & AI — Synthetic Test Data That Actually Catches Bugs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/05_data_generation_ai.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/05_data_generation_ai.ipynb)

Most synthetic data is too clean. It passes every test, ships to prod, and then explodes on a real edge case nobody thought to generate.

LakeLogic's `DataGenerator` reads your contract and produces **realistic data with deliberate edge cases injected** — invalid types, referential gaps, temporal anomalies, boundary values. You can infer contracts from DDL, CSVs, or live tables in seconds, run end-to-end tests, and process unstructured PDFs and tickets through the same contract abstraction.

In [1]:
# Install lakelogic
# Lean install: pdfplumber (PDF extraction, section 7) + spaCy/textblob (NLP, section 7).
# We deliberately do NOT pull `extraction-ocr` — this notebook never uses `unstructured`
# (Office docs) or `rapidocr-onnxruntime` (image OCR), which are large and slow to install.
!pip install -q lakelogic[polars,nlp,pdf]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


lakelogic v1.40.4 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


### ⚙️ Execution Engine

In [2]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

---
## 1. DataGenerator Basics — Synthetic Data From a Contract

**The Problem:** You need test data that matches your schema. Writing Faker scripts for every table is tedious and drifts out of sync with your contracts.

**The Solution:** `DataGenerator` reads your contract and generates realistic data — including controlled invalid rows for quarantine testing.

In [3]:
contract = s.write_contract(
    """
version: 1.0.0
dataset: test_users
model:
  fields:
    - name: user_id
      type: integer
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: age
      type: integer
      min: 18
      max: 120
    - name: country
      type: string
      accepted_values: [US, GB, DE, FR, JP]
    - name: status
      type: string
      accepted_values: [active, inactive, suspended]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: valid_age
      sql: "age BETWEEN 18 AND 120"
    - name: valid_country
      sql: "country IN ('US','GB','DE','FR','JP')"
""",
    "05_data_generation_ai_demo/users.yaml",
)

gen = ll.DataGenerator(contract)
df = gen.generate(rows=1000, invalid_ratio=0.10, output_format=ENGINE)

2026-07-25 06:52:21.815 | INFO     | lakelogic.core.generator:generate:3452 - 📋 Generating data for: test_users
2026-07-25 06:52:21.816 | INFO     | lakelogic.core.generator:generate:3453 -    Records    : 900 valid + 100 invalid = 1,000 total
2026-07-25 06:52:21.817 | INFO     | lakelogic.core.generator:generate:3469 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-07-25 06:52:21.817 | INFO     | lakelogic.core.generator:generate:3484 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-07-25 06:52:21.963 | INFO     | lakelogic.core.generator:generate:3518 -    Row generation complete: 1,000 records built
2026-07-25 06:52:21.965 | INFO     | lakelogic.core.generator:generate:3540 -    Test cases : 214 across 7 categories
2026-07-25 06:52:21.965 | INFO     | lakelogic.core.generator:generate:3542 -      NOT_NULL_VIOLATION               76 injections
2026-07-25 06:52:21.966 | INFO     | lakelogic.core.generator:generate:3542 -      EMPTY_STRING    

In [4]:
# The Proof
print(f"Generated {len(df)} rows with ~10% intentionally invalid")
display(df.head(10))

Generated 1000 rows with ~10% intentionally invalid


user_id,email,age,country,status,_is_invalid,_test_case_types
i64,str,i64,str,str,bool,str
6061,"""krystalyoung@example.org""",null,"""GB""","""active""",false,null
8987,"""cjones@example.com""",18,"""JP""","""inactive""",true,null
4224,"""langjason@example.org""",36,"""DE""","""active""",false,null
null,null,37,"""JP""","""""",true,"""EMPTY_STRING,NOT_NULL_VIOLATION"""
2440,"""baileymakayla@example.org""",30,"""GB""","""suspended""",false,null
3616,"""lcortez@example.org""",35,"""GB""","""suspended""",false,null
2520,"""karensingh@example.net""",24,"""DE""","""active""",false,null
5681,null,null,"""DE""","""active""",true,"""EDGE_CASE_BUILTIN,NOT_NULL_VIOLATION"""
5553,"""jesse37@example.org""",46,"""FR""","""active""",false,null


---
## 2. Custom AI-Steered Scenarios

**The Problem:** Heuristic or pure random data often fails to test very specific business logic states or regional subsets.

**The Solution:** Use `ai=True` alongside `ai_custom_scenario` to specifically guide the AI generator while keeping strict adherence to your schema boundaries.

In [14]:
import os
import lakelogic as ll

# ── 1. Configure the AI Provider ─────────────────────────
# LakeLogic supports: openai, anthropic, azure, Google Gemini, ollama, and anything
# LiteLLM supports.  Set your provider, model, and API key here.
#
# Uncomment the provider you want to use:

# -- OpenAI --
os.environ["LAKELOGIC_AI_PROVIDER"] = "openai"
os.environ["LAKELOGIC_AI_MODEL"] = "gpt-4o-mini"
# os.environ["LAKELOGIC_AI_KEY"]      = "sk-..."

# -- Anthropic --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "anthropic"
# os.environ["LAKELOGIC_AI_MODEL"]    = "claude-sonnet-4-20250514"
# os.environ["ANTHROPIC_API_KEY"]     = "sk-ant-..."

# -- Google Gemini --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "google"
# os.environ["LAKELOGIC_AI_MODEL"]    = "gemini-2.5-flash"
# os.environ["GOOGLE_API_KEY"]         = "AIz....."

# -- Ollama (local, free) --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "ollama"
# os.environ["LAKELOGIC_AI_MODEL"]    = "llama3"

AI_PROVIDER = os.getenv("LAKELOGIC_AI_PROVIDER")
AI_MODEL = os.getenv("LAKELOGIC_AI_MODEL")
AI_API_KEY = os.getenv("LAKELOGIC_AI_KEY", "")  # or ANTHROPIC_API_KEY etc.

# ── 2. Define the data contract ───────────────────────
scenario_contract = s.write_contract(
    """
version: 1.0.0
dataset: ecommerce_users
model:
  fields:
    - name: user_id
      type: integer
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: full_name
      type: string
      required: true
    - name: age
      type: integer
      min: 18
      max: 120
    - name: country
      type: string
      accepted_values: [US, GB, DE, FR, JP, BR, IN]
    - name: tier
      type: string
      accepted_values: [free, pro, enterprise]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: valid_age
      sql: "age BETWEEN 18 AND 120"
""",
    "05_data_generation_ai_demo/scenario_users.yaml",
)

# ── 3. Generate with a custom scenario ───────────────────
# The scenario string is injected directly into the LLM prompt.
# The AI will generate realistic sample pools AND edge cases that
# respect your natural-language instructions.

gen = ll.DataGenerator(scenario_contract)

scenario_df = gen.generate(
    rows=20,
    invalid_ratio=0.20,
    ai=True,
    ai_provider=AI_PROVIDER,
    ai_model=AI_MODEL,
    ai_api_key=AI_API_KEY,
    ai_custom_scenario=(
        "Generate users who are French (FR) or Japanese (JP) only. "
        "All valid users should be enterprise-tier and over 60 years old. "
        "Use realistic French and Japanese names for the full_name field. "
        "For invalid edge cases, inject SQL injection strings into email "
        "and negative ages."
    ),
)

print(f"Generated {s.row_count(scenario_df)} rows with custom AI scenario")
display(s.preview(scenario_df, 10))

# ── 4. Validate through the pipeline ───────────────────
proc = ll.DataProcessor(scenario_contract, engine=ENGINE)
good, bad = proc.run(scenario_df)

print(f"\nGood: {s.row_count(good)} | Quarantined: {s.row_count(bad)}")
s.assert_reconciliation(scenario_df, good, bad)
if s.row_count(bad) > 0:
    print("\nQuarantined rows (AI-generated edge cases):")
    display(s.preview(bad, 5))

2026-07-25 06:57:08.093 | INFO     | lakelogic.ai.provider:get_llm_client:325 - AI provider: openai | model: gpt-4o-mini
2026-07-25 06:57:08.096 | WARNING  | lakelogic.core.generator:generate:3415 - AI data generation failed, falling back to Faker: OpenAI provider requires the 'openai' package. Install with: pip install openai
2026-07-25 06:57:08.097 | INFO     | lakelogic.ai.provider:get_llm_client:325 - AI provider: openai | model: gpt-4o-mini
2026-07-25 06:57:08.098 | WARNING  | lakelogic.core.generator:generate:3437 - AI edge case generation failed: OpenAI provider requires the 'openai' package. Install with: pip install openai
2026-07-25 06:57:08.099 | INFO     | lakelogic.core.generator:generate:3452 - 📋 Generating data for: ecommerce_users
2026-07-25 06:57:08.099 | INFO     | lakelogic.core.generator:generate:3453 -    Records    : 16 valid + 4 invalid = 20 total
2026-07-25 06:57:08.100 | INFO     | lakelogic.core.generator:generate:3469 -    Source     : Faker + heuristic gener

Generated 20 rows with custom AI scenario


,user_id,email,full_name,age,country,tier,_is_invalid,_test_case_types
0,9781,keithjensen@example.com,Christy Holland,39.0,BR,free,False,None
1,5446,ariana64@example.org,David Maldonado,49.0,FR,free,False,None
2,6018,joseph69@example.org,Casey Dickson,34.0,GB,free,False,None
3,6213,heidiperez@example.com,Leslie Benton,38.0,JP,free,False,None
4,1087,brownamanda@example.org,Richard Ward,44.0,US,free,False,None
5,7313,aaron33@example.com,Brian Peterson,43.0,GB,enterprise,False,None
6,7083,None,St. John-Smith,NaN,None,enterprise,True,"EDGE_CASE_BUILTIN,NOT_NULL_VIOLATION"
7,8406,jonespaul@example.net,Lori Peters,NaN,BR,None,True,"EDGE_CASE_BUILTIN,NOT_NULL_VIOLATION"
8,7034,qjennings@example.org,Laura Bennett,36.0,GB,enterprise,False,None
9,6354,jerome14@example.com,Garrett Ortega,57.0,DE,pro,False,None


2026-07-25 06:57:08.178 | WARNING  | lakelogic.core.masking_engine:apply:381 - PII fields detected without masking strategy: [email]. Set 'masking:' (nullify|hash|redact|partial|encrypt) in your contract to enable masking for these fields.
2026-07-25 06:57:08.179 | INFO     | lakelogic.core.masking_engine:apply:388 - No PII fields with explicit masking strategy — skipping masking.
2026-07-25 06:57:08.179 | INFO     | lakelogic.core.processor:run:980 - Run complete | Source: 20 | Total: 20 | Good: 17 | Quarantine: 3 | Ratio: 15.00%
2026-07-25 06:57:08.180 | WARNING  | lakelogic.core.processor:run:1188 - Schema drift detected for 'ecommerce_users': missing=[], unknown=['_is_invalid', '_test_case_types']



Good: 17 | Quarantined: 3
source=20  good=17  bad=3
20 == 17 + 3 -> True

Quarantined rows (AI-generated edge cases):


,user_id,email,full_name,age,country,tier,_is_invalid,_test_case_types,_lakelogic_errors,_lakelogic_categories,quarantine_state,quarantine_reprocessed
0,7083,None,St. John-Smith,NaN,None,enterprise,True,"EDGE_CASE_BUILTIN,NOT_NULL_VIOLATION","[Rule failed: email_required (""email"" IS NOT NULL), Rule failed: valid_email (email LIKE '%@%.%'), Rule failed: valid_age (age BETWEEN 18 AND 120)]","[completeness, correctness, correctness]",active,False
1,8406,jonespaul@example.net,Lori Peters,NaN,BR,None,True,"EDGE_CASE_BUILTIN,NOT_NULL_VIOLATION",[Rule failed: valid_age (age BETWEEN 18 AND 120)],[correctness],active,False
2,1170,jamestyler@example.org,Matthew Gilbert,NaN,GB,enterprise,False,None,[Rule failed: valid_age (age BETWEEN 18 AND 120)],[correctness],active,False


---
## 3. Streaming Simulation -- Time-Windowed Batch Generation

**The Problem:** Your pipeline must handle incremental data arriving in time windows.
You need test data that simulates realistic ingestion patterns -- not just static dumps.

**The Solution:** `DataGenerator.generate_stream()` produces batches with monotonically
increasing timestamps, each confined to a configurable time window.


In [ ]:
# -- Streaming Simulation: time-windowed batch generation ----------
import lakelogic as ll

stream_contract = s.write_contract(
    """
version: 1.0.0
dataset: streaming_events

model:
  fields:
    - name: event_id
      type: integer
    - name: event_ts
      type: timestamp
    - name: user_id
      type: integer
    - name: action
      type: string
      accepted_values: [page_view, click, purchase, signup]
""",
    "05_data_generation_ai_demo/streaming_events.yaml",
)

gen = ll.DataGenerator(stream_contract)

# Simulate 3 batches arriving every 15 minutes, 5 rows each
all_rows = []
for window_start, window_end, batch_df in gen.generate_stream(batches=3, interval_minutes=15, rows_per_batch=5):
    print(f"Window: {window_start} -> {window_end} | {s.row_count(batch_df)} rows")
    all_rows.extend(s.to_records(batch_df))

print(f"\nTotal: {len(all_rows)} events across 3 windows")
display(s.preview(s.to_frame(all_rows), 10, columns=["event_id", "event_ts", "action"]))
print("\n✅ Streaming simulation. Faker. $0 cost.")

---
## 4. Referential Integrity — FK/PK Consistency Across Tables

**The Problem:** You generate test customers and test orders separately. Half your order rows reference `customer_id` values that don't exist in the customers table. Your join tests fail for the wrong reasons.

**The Solution:** `DataGenerator.generate_related()` detects FK/PK relationships between contracts, generates parent tables first, then passes parent PKs into child tables so every foreign key is valid.

In [ ]:
# Define two related contracts: customers (parent) and orders (child)
customers_path = s.write_contract(
    """
version: 1.0.0
dataset: customers
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: tier
      type: string
      accepted_values: [free, pro, enterprise]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
  dataset_rules:
    - unique: customer_id
""",
    "05_data_generation_ai_demo/ri_customers.yaml",
)

orders_path = s.write_contract(
    """
version: 1.0.0
dataset: orders
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: customer_id
      type: integer
      required: true
      foreign_key:
        contract: customers
        column: customer_id
    - name: amount
      type: float
      required: true
    - name: status
      type: string
      accepted_values: [pending, shipped, delivered]
quality:
  row_rules:
    - name: positive_amount
      sql: "amount > 0"
    - referential_integrity:
        field: customer_id
        contract: customers
        column: customer_id
        severity: critical
  dataset_rules:
    - unique: order_id
""",
    "05_data_generation_ai_demo/ri_orders.yaml",
)

# Generate both tables with referential integrity
related = ll.DataGenerator.generate_related(
    contracts={
        "customers": customers_path,
        "orders": orders_path,
    },
    rows={"customers": 50, "orders": 200},
    invalid_ratio=0.05,
)

customers_df = related["customers"]
orders_df = related["orders"]

In [ ]:
# The Proof — every order.customer_id exists in customers.customer_id
from collections import Counter

parent_ids = {r["customer_id"] for r in s.to_records(customers_df)}
child_records = s.to_records(orders_df)
child_ids = {r["customer_id"] for r in child_records}
orphans = child_ids - parent_ids

print(f"Customers: {s.row_count(customers_df)} rows, {len(parent_ids)} unique IDs")
print(f"Orders:    {s.row_count(orders_df)} rows")
print(f"Orphan FKs: {len(orphans)}")
print()

# Show the FK distribution (top 5 customers by order count)
fk_counts = Counter(r["customer_id"] for r in child_records).most_common(5)
print("Orders per customer (top 5):")
display(s.preview(s.to_frame([{"customer_id": k, "orders": v} for k, v in fk_counts])))
print("\n50 customers, 200 orders — every FK references a real parent row.")

In [ ]:
# Validate both tables through their contracts
proc_c = ll.DataProcessor(customers_path, engine=ENGINE)
good_c, bad_c = proc_c.run(customers_df)

proc_o = ll.DataProcessor(orders_path, engine=ENGINE)
good_o, bad_o = proc_o.run(orders_df)

print("Customers:")
s.assert_reconciliation(customers_df, good_c, bad_c)
print("\nOrders:")
s.assert_reconciliation(orders_df, good_o, bad_o)
print("\nBoth tables validated. Referential integrity preserved end-to-end.")

### 4b. Explicit Relationships — When Column Names Differ

The example above worked because both tables share the column name `customer_id`. But what if the parent uses `id` as its primary key and the child uses `cust_id` as its foreign key?

Use the `relationships` parameter to explicitly declare the FK→PK mapping. This also works with **DDL strings**, **tuple lists**, and **dicts** — no YAML files needed.

In [ ]:
# ── 3-Table Diamond: Different PK/FK Column Names ───────────────────
# Parent tables have PK "id", child table has FK "cust_id" and "prod_id"

related = ll.DataGenerator.generate_related(
    contracts={
        "customers": "id BIGINT, name STRING, email STRING",
        "products": "id BIGINT, product_name STRING, price DOUBLE",
        "sales": "sale_id BIGINT, cust_id BIGINT, prod_id BIGINT, amount DOUBLE",
    },
    rows={"customers": 20, "products": 10, "sales": 100},
    relationships=[
        {"child": "sales", "child_column": "cust_id", "parent": "customers", "parent_column": "id"},
        {"child": "sales", "child_column": "prod_id", "parent": "products", "parent_column": "id"},
    ],
    invalid_ratio=0.05,
)

# Verify referential integrity
cust_ids = set(related["customers"]["id"].to_list())
prod_ids = set(related["products"]["id"].to_list())
sale_cust = set(related["sales"]["cust_id"].to_list())
sale_prod = set(related["sales"]["prod_id"].to_list())

print(f"Customers: {len(related['customers'])} rows ({len(cust_ids)} unique IDs)")
print(f"Products:  {len(related['products'])} rows ({len(prod_ids)} unique IDs)")
print(f"Sales:     {len(related['sales'])} rows")
print(f"\nOrphan cust_id FKs: {len(sale_cust - cust_ids)}")
print(f"Orphan prod_id FKs: {len(sale_prod - prod_ids)}")
print("\n3 tables, 2 explicit FK mappings, zero orphans.")
print("Column names differ (id vs cust_id/prod_id) — explicit relationships handle it.")
display(related["sales"].head(5))

### 4c. Simulating Late Arriving Data (Orphan FKs)

**The Problem:** Referential integrity failures in production don't always look like random garbage; they look like perfectly valid IDs that just haven't arrived in the parent table yet (e.g. late arriving dimensions).

**The Solution:** Use `invalid_ratio > 0` with `generate_related()`. When LakeLogic targets an FK column for invalidation, it intentionally corrupts the ID to guarantee an orphan record failure (e.g. appending `_ORPHAN` to strings or offsetting numeric IDs).

In [ ]:
related_with_orphans = ll.DataGenerator.generate_related(
    contracts={
        "customers": "id BIGINT, name STRING",
        "orders": "order_id BIGINT, cust_id BIGINT, amount DOUBLE",
    },
    rows={"customers": 10, "orders": 100},
    relationships=[
        {"child": "orders", "child_column": "cust_id", "parent": "customers", "parent_column": "id"},
    ],
    invalid_ratio=0.05,  # 5% of generated rows will be deliberately invalid
)

orders = related_with_orphans["orders"]
invalid_orders = orders.filter(orders["_is_invalid"] == True)

# Prove they are missing from the parent table
parent_ids = set(related_with_orphans["customers"]["id"].to_list())
child_ids = set(invalid_orders["cust_id"].to_list())

print(f"Orphan IDs generated: {child_ids - parent_ids}\n")
print("Notice how integer IDs are systematically shifted (+999000) to ensure they are orphaned")
print("while remaining valid data types, perfectly simulating late-arriving dimension records.")
display(invalid_orders.head(5))

---
## 5. `infer_contract` from Schema — No Data Needed

**The Problem:** You know the table schema (from Spark, a DDL script, or a spec doc) but don't have sample data yet. Writing the contract YAML by hand is tedious and error-prone.

**The Solution:** Pass a **DDL string**, **Spark StructType**, **list of tuples**, or **dict** directly to `infer_contract` — it generates the full contract with correct types and PII detection from column names alone.

In [ ]:
from lakelogic.core.bootstrap import infer_contract

# ── From a Spark DDL string (the most common format) ─────────────
draft = infer_contract(
    "order_id BIGINT, customer_email STRING, amount DECIMAL(10,2), status STRING, created_at TIMESTAMP",
    title="Orders",
    domain="commerce",
)
draft.show()
print()
print("PII auto-detected: customer_email flagged as email")
print("Types mapped: BIGINT -> integer, DECIMAL -> double, TIMESTAMP -> timestamp")

In [ ]:
# ── From a list of (name, type) tuples ────────────────────────
# Great for programmatic contract creation
draft_tuples = infer_contract(
    [
        ("patient_id", "string"),
        ("ssn", "string"),
        ("diagnosis_code", "string"),
        ("admission_date", "date"),
        ("discharge_date", "date"),
        ("total_cost", "double"),
    ],
    title="Patient Records",
    domain="healthcare",
)
draft_tuples.show()
print()
print("PII auto-detected: ssn flagged as SSN")
print("Temporal pair: admission_date <= discharge_date would be suggested with data")

In [ ]:
# ── DataGenerator from schema directly ──────────────────────
# No contract YAML needed. No CSV needed. Just the schema.
import lakelogic as ll

# DDL string → synthetic data in 2 lines
gen = ll.DataGenerator("order_id BIGINT, customer_email STRING, amount DOUBLE, status STRING, created_at TIMESTAMP")
df = gen.generate(rows=10, invalid_ratio=0.10, output_format=ENGINE)
print("DDL string → DataGenerator → 10 rows (with 10% intentionally invalid):")
display(df)

print()

# List of tuples → synthetic data
gen2 = ll.DataGenerator(
    [
        ("patient_id", "string"),
        ("ssn", "string"),
        ("diagnosis_code", "string"),
        ("admission_date", "date"),
        ("total_cost", "double"),
    ]
)
df2 = gen2.generate(rows=5, output_format=ENGINE)
print("Tuple list → DataGenerator → 5 rows:")
display(df2)

print()

# Dict → synthetic data
gen3 = ll.DataGenerator({"product_id": "integer", "name": "string", "price": "decimal", "in_stock": "boolean"})
df3 = gen3.generate(rows=5, output_format=ENGINE)
print("Dict → DataGenerator → 5 rows:")
display(df3)

---
## 5b. End-to-End Testing (Schema → Data → LakeLogic)

**The Problem:** You want to write a unit test for a pipeline, proving that the contract correctly quarantines bad rows, but you don't want to maintain test CSV files.

**The Solution:** Use the schema shorthand to spin up a generator, pump out data with intentional failures, and run it through LakeLogic in 5 lines of code.

In [ ]:
import lakelogic as ll
from lakelogic.core.bootstrap import infer_contract

# 1. Define schema in 1 line & generate data
gen = ll.DataGenerator("order_id BIGINT, email STRING, amount DECIMAL(10,2)")

# 2. Generate 100 test rows with 15% intentional failures
df = gen.generate(rows=100, invalid_ratio=0.15, output_format=ENGINE)

# 3. Infer a contract from the SAME schema
contract = infer_contract("order_id BIGINT, email STRING, amount DECIMAL(10,2)")
contract.save("test_contract.yaml")

# 4. Run the test case — prove quarantine catches the bad rows
proc = ll.DataProcessor("test_contract.yaml", engine=ENGINE)
res = proc.run(df)

print(f"\nTest passed! Total rows processed: {res.source_count}")
print(f"Valid rows sent to Catalog: {res.good_count}")
print(f"Invalid rows Quarantined: {res.bad_count}")

---
## 5c. Inferring Directly from Unity Catalog or Database Tables

If you already have a live table, you can infer a contract and generate data straight from the catalog.

In [ ]:
from lakelogic.core.bootstrap import infer_contract

# To infer a contract from a live Databricks Unity Catalog table:
#
# draft = infer_contract("my_catalog.sales.orders")
# draft.save("contracts/orders.yaml")

# To chain directly into generating synthetic test data:
#
# gen = infer_contract("my_catalog.sales.orders").to_generator(seed=42)
# df = gen.generate(rows=500, invalid_ratio=0.05, output_format=ENGINE)
# display(df)

---
## 6. `infer_contract` — Contract From a CSV in 30 Seconds

**The Problem:** You have 50 CSVs and no contracts. Writing YAML by hand for each one takes days.

**The Solution:** Point `infer_contract` at a file. It detects types, PII fields, and suggests quality rules.

In [ ]:
from lakelogic.core.bootstrap import infer_contract

# Create a sample CSV (built engine-agnostically from records)
_countries = ["US", "GB", "DE", "FR", "JP"]
sample_rows = [
    {
        "order_id": i,
        "customer_email": f"user{i}@example.com",
        "amount": round(i * 9.99, 2),
        "country": _countries[(i - 1) % 5],
        "created_at": "2026-01-15",
    }
    for i in range(1, 101)
]
s.to_frame(sample_rows).write_csv("sample_orders.csv")

# Infer a contract from the CSV
draft = infer_contract("sample_orders.csv", title="Inferred Orders")

In [ ]:
# The Proof
draft.show()
print("\nContract inferred in seconds. PII detected. Types resolved. Ready to customise.")

---
## 7. Unstructured Processing — Contract-Driven Extraction

**The Problem:** You have PDFs, scanned images, or free-text. Regex breaks on format changes. Custom parsers drift from your schema.

**The Solution:** Declare *what* to extract in `model.fields` and *how* in `extraction:`. LakeLogic picks the right library, extracts, validates, and materialises — one call.

| Provider | Extra | Input | Use Case |
|----------|-------|-------|----------|
| `local` (pdfplumber) | `lakelogic[pdf]` | PDF | Table + text, $0, no API key |
| `spacy` | `lakelogic[nlp]` | Free text | NER + classification, $0, local |
| `rapidocr` | `lakelogic[extraction-ocr]` | Scanned image | ONNX OCR, pure Python, no torch |
| `openai` / `anthropic` | `lakelogic[ai]` | Any | LLM prompting with structured output |

> This notebook installs only `lakelogic[polars,nlp,pdf]` — the lean set the cells below actually use. If you want the scanned-image (`rapidocr`) or Office-doc (`unstructured`) providers, install `lakelogic[extraction-ocr]` instead.

In [ ]:
# Generate demo assets — invoice PDF and support tickets
import os
import shutil
import tempfile
import subprocess
import sys

DEMO_DIR = os.path.join(tempfile.gettempdir(), "lakelogic_extraction_demo")
shutil.rmtree(DEMO_DIR, ignore_errors=True)
os.makedirs(DEMO_DIR, exist_ok=True)

try:
    from fpdf import FPDF
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fpdf2"])
    from fpdf import FPDF

# Build a realistic invoice PDF with a table of line items
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", "B", 20)
pdf.cell(0, 15, "INVOICE", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.set_font("Helvetica", "", 11)
pdf.cell(0, 8, "Acme Corp | 500 Market St, San Francisco, CA 94105", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.ln(4)
pdf.set_font("Helvetica", "B", 11)
pdf.cell(95, 8, "Invoice #: INV-2026-0042")
pdf.cell(95, 8, "Date: April 15, 2026", new_x="LMARGIN", new_y="NEXT", align="R")
pdf.cell(95, 8, "Bill To: Globex Corporation")
pdf.cell(95, 8, "Due: May 15, 2026", new_x="LMARGIN", new_y="NEXT", align="R")
pdf.ln(4)
pdf.set_fill_color(240, 240, 240)
pdf.set_font("Helvetica", "B", 10)
for col, w in [("Description", 90), ("Hours", 30), ("Rate", 35), ("Amount", 35)]:
    is_last = col == "Amount"
    pdf.cell(w, 8, col, border=1, fill=True, align="C", **(dict(new_x="LMARGIN", new_y="NEXT") if is_last else {}))
pdf.set_font("Helvetica", "", 10)
LINE_ITEMS = [
    ("Data Platform Architecture", 40, 200, 8000),
    ("Pipeline Development", 60, 175, 10500),
    ("Quality Assurance & Testing", 20, 150, 3000),
]
for desc, hrs, rate, amt in LINE_ITEMS:
    pdf.cell(90, 7, desc, border=1)
    pdf.cell(30, 7, str(hrs), border=1, align="C")
    pdf.cell(35, 7, f"${rate:.2f}", border=1, align="C")
    pdf.cell(35, 7, f"${amt:,.2f}", border=1, align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "B", 12)
pdf.cell(155, 10, "Total:", align="R")
pdf.cell(35, 10, "$21,500.00", align="C", new_x="LMARGIN", new_y="NEXT")

PDF_PATH = os.path.join(DEMO_DIR, "demo_invoice.pdf")
pdf.output(PDF_PATH)

# Five support tickets for classification
_ticket_bodies = [
    "I was charged $2,500 for a subscription I cancelled. Billing error ongoing since March.",
    "Order #4521 shipped to London but I live in Manchester. Please redirect via FedEx.",
    "New MacBook Pro has a cracked screen. Returns team arranged replacement immediately.",
    "Enterprise license for 500 seats expires next week. Discuss renewal and 200 more seats.",
    "API returning 500 errors since 2pm. Blocking our entire production pipeline. Fix now.",
]
tickets = s.to_frame([{"ticket_id": 1001 + i, "ticket_body": b} for i, b in enumerate(_ticket_bodies)])


print(f"PDF invoice : {PDF_PATH}")

In [ ]:
# -- Flavour 1: PDF Invoice → pdfplumber --------------------------
# The contract defines model fields, extraction provider, and quality rules.
# extract_file() handles all parsing — no manual logic in the notebook.

from lakelogic.engines.llm import extract_file
from lakelogic.core.models import ExtractionConfig

pdf_contract = s.write_contract(
    """
version: 1.0.0
dataset: invoice_line_items

model:
  fields:
    # -- Metadata (extracted from page text via regex) ----------
    - name: invoice_number
      type: string
      required: true
      extraction_task: metadata
      extraction_examples: ['Invoice #:\\s*(\\S+)']
    - name: vendor
      type: string
      extraction_task: metadata
      extraction_examples: ['INVOICE\\n(.+?)\\n']
    - name: date
      type: string
      extraction_task: metadata
      extraction_examples: ['Date:\\s*(.+?)\\n']
    - name: bill_to
      type: string
      extraction_task: metadata
      extraction_examples: ['Bill To:\\s*(.+?)\\s+Due:']
    - name: due_date
      type: string
      extraction_task: metadata
      extraction_examples: ['Due:\\s*(.+?)(?:\\n|$)']

    # -- Table rows (matched by column header name) ------------
    - name: description
      type: string
      required: true
    - name: hours
      type: integer
    - name: rate
      type: string
    - name: amount
      type: string

extraction:
  provider: pdfplumber

quality:
  row_rules:
    - name: has_description
      sql: "description IS NOT NULL AND description != ''"
    - name: has_invoice_number
      sql: "invoice_number IS NOT NULL"
""",
    "05_data_generation_ai_demo/invoice_line_items.yaml",
)

# -- Extract -------------------------------------------------------
import yaml

contract_dict = yaml.safe_load(open(pdf_contract))
ext_config = ExtractionConfig(
    provider=contract_dict["extraction"]["provider"],
    output_schema=contract_dict["model"]["fields"],
)
rows = extract_file(PDF_PATH, ext_config)

# -- Materialize --------------------------------------------------
df_invoice = s.to_frame(rows)


display(s.preview(df_invoice))
print(f"\n\u2705 PDF \u2192 {len(rows)} line items + metadata. pdfplumber. $0 cost.")

In [ ]:
# ── Side-by-side: Raw PDF text vs Extracted Table ─────────────────
import pdfplumber

with pdfplumber.open(PDF_PATH) as doc:
    raw_text = doc.pages[0].extract_text()

print("RAW PDF TEXT".center(60, "\u2500"))
print(raw_text)
print()
print("EXTRACTED TABLE".center(60, "\u2500"))
display(df_invoice.drop([c for c in df_invoice.columns if c.startswith("_")]))
print(f"\n\u2500 Source: {PDF_PATH}")

In [ ]:
# -- Flavour 2: Support Tickets -> spaCy NER + Classification ----
# spaCy extracts entities + classifies text locally at production speed.

from lakelogic.engines.llm import extract_row

nlp_contract = s.write_contract(
    """
version: 1.0.0
dataset: enriched_tickets

model:
  fields:
    - name: persons
      type: string
      extraction_task: ner
    - name: organizations
      type: string
      extraction_task: ner
      extraction_examples: [ORG]
    - name: category
      type: string
      extraction_task: classification
      accepted_values: [billing, shipping, product, enterprise, outage]
    - name: sentiment
      type: string
      extraction_task: sentiment

extraction:
  provider: spacy
  model: en_core_web_sm       # small model -- fast, tiny download; swap to _md/_lg for better NER
  text_column: ticket_body

quality:
  row_rules:
    - name: has_sentiment
      sql: "sentiment IS NOT NULL"
""",
    "05_data_generation_ai_demo/enriched_tickets.yaml",
)

# -- Extract each ticket -----------------------------------------------
contract_dict = yaml.safe_load(open(nlp_contract))
ext_config = ExtractionConfig(
    provider=contract_dict["extraction"]["provider"],
    text_column=contract_dict["extraction"].get("text_column", "text"),
    output_schema=contract_dict["model"]["fields"],
)

enriched = [extract_row(row, ext_config) for row in tickets.to_dicts()]

# -- Materialize -------------------------------------------------------
df_tickets = s.to_frame(enriched)

display_cols = ["ticket_id", "persons", "organizations", "category", "sentiment"]

print("RAW ticket TEXT".center(60, "\u2500"))
display(s.preview(tickets))
print()
print("EXTRACTED TABLE".center(60, "\u2500"))

display(s.preview(df_tickets, columns=display_cols))
print(f"\n\u2705 {len(enriched)} tickets enriched. spaCy. $0 cost.")

---
## 8. Automated Run Logs — Structured Pipeline Observability

**The Problem:** Pipelines fail silently. Row counts drift. Quarantine tables fill up. But you only find out when a dashboard is empty.

**The Solution:** Every pipeline run automatically emits a structured, comprehensive run log. These logs can be written out to a Delta table, making your entire data operations history immediately queryable.

In [ ]:
import lakelogic as ll
from lakelogic.core.run_log import write_run_log
import duckdb
import os
import tempfile

# ── 1. Configure the pipeline to write actual run logs to DuckDB ──────
LOG_DIR = os.path.join(tempfile.gettempdir(), "lakelogic_logs")
os.makedirs(LOG_DIR, exist_ok=True)
DB_PATH = os.path.join(LOG_DIR, "run_logs.duckdb").replace("\\", "/")

# Remove stale DB so we start fresh each demo run
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

log_contract_path = s.write_contract(
    f"""
version: 1.0.0
dataset: automated_log_demo
metadata:
  run_log_table: pipeline_run_logs
  run_log_backend: duckdb
  run_log_database: "{DB_PATH}"
model:
  fields:
    - name: user_id
      type: integer
    - name: age
      type: integer
quality:
  row_rules:
    - name: valid_age
      sql: "age >= 18"
""",
    "05_data_generation_ai_demo/automated_log_demo.yaml",
)

# Load as a proper DataContract object (write_run_log needs .metadata)
log_contract = ll.DataContract.from_yaml(log_contract_path)

# ── 2. Run the pipeline a few times with varying data quality ─────────
proc = ll.DataProcessor(log_contract, engine=ENGINE)


def simulate_run(data):
    proc.run(s.to_frame(data, engine=ENGINE))
    # Status is normally set by the pipeline runner after materialize;
    # in standalone mode we stamp it manually.
    report = proc.last_report
    quarantined = (report.get("counts") or {}).get("quarantined", 0)
    report["status"] = "warning" if quarantined else "success"
    write_run_log(report, log_contract)


# Run 1: Perfect data
simulate_run({"user_id": [1, 2], "age": [25, 30]})

# Run 2: One invalid row (age < 18)
simulate_run({"user_id": [3, 4], "age": [15, 40]})

# Run 3: All invalid rows
simulate_run({"user_id": [5, 6], "age": [10, 12]})

# ── 3. Query the actual Run Logs telemetry table ──────────────────────
con = duckdb.connect(DB_PATH, read_only=True)
query = """
  SELECT 
      run_id,
      contract,
      dataset, 
      counts_source, 
      counts_good, 
      counts_quarantined, 
      quarantine_ratio,
      status,
      start_time, 
      end_time,
      run_duration_seconds
  FROM pipeline_run_logs
"""
logs_df = con.execute(query).df()
con.close()

print("AUTOMATED RUN LOGS (Queried from actual DuckDB backend):")
display(logs_df)

print(f"\n\u2705 {len(logs_df)} runs captured. BI tools can connect directly to: {DB_PATH}")

## What You Just Did

Eight building blocks that normally take weeks of test-data engineering:

- ✅ **Synthetic data from contracts** — `DataGenerator` with edge cases baked in
- ✅ **AI-steered scenarios** — domain-specific test cases on demand
- ✅ **Streaming simulation** — time-windowed batch generation
- ✅ **Referential integrity** — multi-table FK/PK consistency, even with orphans
- ✅ **`infer_contract` from DDL or CSVs** — 30 seconds from schema to contract
- ✅ **End-to-end test scaffolding** — schema → data → validation in one file
- ✅ **Unstructured processing** — PDFs and tickets through the same contract abstraction
- ✅ **Structured run logs** — every pipeline run observable by default

Hours saved on manufacturing test data: **a lot**.

---
## Go Deeper — Explore by Capability

Each notebook below is **self-contained** and maps to one pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities). Pick the one that matters to you most.

| # | Notebook | What You'll See |
|---|---|---|
| 🚀 | **[Quickstart](00_quickstart.ipynb)** | One contract, every row accounted for, PII masked — in 5 minutes |
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

---

**Like what you saw?** ⭐ [Star us on GitHub](https://github.com/LakeLogic/LakeLogic) — it's how we know this matters.